# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display, DisplayHandle
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")

if api_key and api_key.startswith("AQ") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'gemini-flash-lite-latest'
openai = OpenAI(
    api_key=api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

API key looks good so far


In [3]:
links = fetch_website_links("https://ithmaar-solutions.com/index.html")
links

['index.html',
 'about.html',
 'about.html',
 'our-team.html',
 'why-ithmaar.html',
 'Ithmaar-Finance-overview.html',
 'Ithmaar-Finance-overview.html',
 'ithmaar-finance-modules.html',
 'ithmaarfinancefeatures.html',
 'ithmaar-finance-process.html',
 'ithmaar-ai.html',
 'ithmaar-finance-tech.html',
 'consultancy-services.html',
 'contact-us.html',
 '#',
 'about.html',
 'Ithmaar-Finance-overview.html',
 'ithmaar-ai.html',
 'contact-us.html',
 'ithmaar-finance-modules.html',
 'Ithmaar-Finance-overview.html',
 'our-team.html',
 'ithmaar-ai.html',
 'index.html',
 'about.html',
 'Ithmaar-Finance-overview.html',
 'ithmaar-ai.html',
 'ithmaar-finance-tech.html',
 'contact-us.html',
 '#',
 '#',
 'https://www.linkedin.com/company/ithmaarsolutions',
 'https://www.youtube.com/channel/UC8kiIu8015yTZMX_bmSRBZg']

## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,           
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(str(link) for link in links)
    return user_prompt  # <--- Add this return statement

In [6]:
print(get_links_user_prompt("https://ithmaar-solutions.com/index.html"))


Here is the list of links on the website https://ithmaar-solutions.com/index.html -
Please decide which of these are relevant web links for a brochure about the company,           
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

index.html
about.html
about.html
our-team.html
why-ithmaar.html
Ithmaar-Finance-overview.html
Ithmaar-Finance-overview.html
ithmaar-finance-modules.html
ithmaarfinancefeatures.html
ithmaar-finance-process.html
ithmaar-ai.html
ithmaar-finance-tech.html
consultancy-services.html
contact-us.html
#
about.html
Ithmaar-Finance-overview.html
ithmaar-ai.html
contact-us.html
ithmaar-finance-modules.html
Ithmaar-Finance-overview.html
our-team.html
ithmaar-ai.html
index.html
about.html
Ithmaar-Finance-overview.html
ithmaar-ai.html
ithmaar-finance-tech.html
contact-us.html
#
#
https://www.linkedin.com/company/ithmaarsolutions
https://www.youtube.com/channel/UC8kiIu8015yTZMX_bmSR

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ], 
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content or ""
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://ithmaar-solutions.com/index.html")

{'links': [{'type': 'about page',
   'url': 'https://ithmaar-solutions.com/about.html'},
  {'type': 'team page', 'url': 'https://ithmaar-solutions.com/our-team.html'},
  {'type': 'company info',
   'url': 'https://ithmaar-solutions.com/why-ithmaar.html'},
  {'type': 'services page',
   'url': 'https://ithmaar-solutions.com/consultancy-services.html'},
  {'type': 'contact page',
   'url': 'https://ithmaar-solutions.com/contact-us.html'},
  {'type': 'social media',
   'url': 'https://www.linkedin.com/company/ithmaarsolutions'},
  {'type': 'social media',
   'url': 'https://www.youtube.com/channel/UC8kiIu8015yTZMX_bmSRBZg'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content or ""
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-flash-lite-latest
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-flash-lite-latest
Found 7 relevant links


{'links': [{'type': 'company page',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand assets', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'github', 'url': 'https://github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://ithmaar-solutions.com/index.html"))

Selecting relevant links for https://ithmaar-solutions.com/index.html by calling gemini-flash-lite-latest
Found 7 relevant links
## Landing Page:

ITHMAAR

Loading...
HOME
(current)
ABOUT
About us
Our Team
WHY ITHMAAR
ITHMAAR FINANCE
ITHMAAR FINANCE
Overview
ITHMAAR FINANCE
modules
ITHMAAR FINANCE
Features
ITHMAAR FINANCE
process
ITHMAAR AI
TECHNOLOGY
CONSULTANCY SERVICES
CONTACT
                US
THE FUTURE,
YOUR WAY.
Optimizing the
                            experience of every
                            organization and individual within the Islamic financial sector through cutting edge
                            technology.
Hear Our
                            Story
A NEW GENERATION OF ISLAMIC
                            FINANCE SYSTEMS
Introducing a new
                            generation
of Islamic Finance Systems
Explore Ithmaar Finance
PRECISE PREDICTIONS WITH ITHMAAR AI
Ithmaar Artificial Intelligence technologies
add precise predictions to your business to fuel
the
   

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("Ithmaar", "https://ithmaar-solutions.com/index.html")

Selecting relevant links for https://ithmaar-solutions.com/index.html by calling gemini-flash-lite-latest
Found 8 relevant links


"\nYou are looking at a company called: Ithmaar\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nITHMAAR\n\nLoading...\nHOME\n(current)\nABOUT\nAbout us\nOur Team\nWHY ITHMAAR\nITHMAAR FINANCE\nITHMAAR FINANCE\nOverview\nITHMAAR FINANCE\nmodules\nITHMAAR FINANCE\nFeatures\nITHMAAR FINANCE\nprocess\nITHMAAR AI\nTECHNOLOGY\nCONSULTANCY SERVICES\nCONTACT\n                US\nTHE FUTURE,\nYOUR WAY.\nOptimizing the\n                            experience of every\n                            organization and individual within the Islamic financial sector through cutting edge\n                            technology.\nHear Our\n                            Story\nA NEW GENERATION OF ISLAMIC\n                            FINANCE SYSTEMS\nIntroducing a new\n                            generation\nof Islamic Finance Systems\nExplore Ithmaar Finance\nPRECISE P

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemini-flash-lite-latest",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("Ithmaar", "https://ithmaar-solutions.com/index.html")

Selecting relevant links for https://ithmaar-solutions.com/index.html by calling gemini-flash-lite-latest
Found 8 relevant links


# Ithmaar Integrated Solutions

## The Future, Your Way

Welcome to Ithmaar, a pioneer in the world of Islamic FinTech. We bring state-of-the-art technologies to tackle the industry's biggest challenges, reshaping the future of Islamic Finance and optimizing the experience of every organization and individual within the sector. 


## Who We Are

Founded by a team of experienced business and technology experts—industry veterans with decades of collective experience—Ithmaar was built with a singular vision: to empower financial organizations to thrive in the face of rapid change. 

Our leadership and team bring deep expertise spanning Islamic FinTech, microfinance, retail/consumer finance, and corporate finance. Operating under a core belief of 'no compromises,' we build integrated, modular systems that facilitate seamless transitions from legacy platforms, allowing institutions to choose only the modules they need for optimal efficiency.


## Our Offerings: Cutting-Edge Islamic Finance

Ithmaar provides robust, flexible, and 100% Sharia-compliant solutions designed to navigate the complexities of modern finance. 

* **Ithmaar Finance:** A full-fledged, next-generation Islamic finance system covering all finance sectors (Retail, Corporate, and more) and the entire business process. Packed with all the necessary tools and functions, it enables you to provide exceptional service to your customers. Recent innovations include our Tejara product for managing supply chain receivables factoring.
* **Ithmaar AI:** Our advanced Artificial Intelligence technologies add precise predictions to your business operations, fueling optimum strategic decisions and unleashing new horizons to pursue.
* **Consultancy Services & Technology:** Tailored guidance and robust tech infrastructure designed to elevate financial institutions.


## Our Culture

At Ithmaar, we foster a culture of innovation, adaptability, and excellence. Driven by a game-changing mentality and a unique vision, our team is dedicated to revolutionizing the way financial institutions grow their business. We value forward-thinking minds who are passionate about reshaping traditional paradigms and delivering unparalleled functionality and user experiences in the financial sector.


## Join Us & Partner With Us

**For Customers:** Experience a new generation of Islamic finance systems tailored to your unique requirements. Request a demo today and explore the full potential of Ithmaar Finance.

**For Investors & Recruits:** Join a high-impact, visionary team of industry veterans and tech experts. Partner with us as we scale new heights in Islamic FinTech, bringing precision, intelligence, and modern efficiency to financial institutions worldwide.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [32]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-3.8-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)},
        ],
        stream=True,
    )
    response = ""
    from IPython.display import DisplayHandle
    display_handle = DisplayHandle()
    display_handle.display(Markdown(""))
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        display_handle.update(Markdown(response))

In [34]:
stream_brochure("Ithmaar", "https:// ithmaar-solutions.com/index.html")

ConnectionError: HTTPSConnectionPool(host='%20ithmaar-solutions.com', port=443): Max retries exceeded with url: /index.html (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x1133d1f10>: Failed to resolve '%20ithmaar-solutions.com' ([Errno 8] nodename nor servname provided, or not known)"))

In [35]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Ithmaar", "https://ithmaar-solutions.com/index.html")

Selecting relevant links for https://ithmaar-solutions.com/index.html by calling gemini-flash-lite-latest
Found 7 relevant links


# Ithmaar Integrated Solutions
**The Future, Your Way: Reshaping the Future of Islamic Finance**

Ithmaar Integrated Solutions is a next-generation financial technology company dedicated to modernizing and optimizing the Islamic financial sector. By merging deep domain expertise with cutting-edge software and artificial intelligence, Ithmaar empowers financial institutions to navigate modern complexities, replace legacy limitations, and thrive in an evolving global market.

---

### What We Do: Products & Innovation

Ithmaar delivers high-performance, modular software systems designed for agility, scale, and uncompromising compliance:

* **Ithmaar Finance:** A full-fledged, multi-dimensional finance system covering the entire business lifecycle across microfinance, retail/consumer finance, and corporate banking. Built with an adaptable architecture, it supports both Islamic and conventional models while facilitating seamless transitions from legacy platforms.
* **Ithmaar AI:** Advanced artificial intelligence and predictive modeling tools that empower executives with data-driven foresight, optimizing strategic decision-making and risk management.
* **Tejara:** A dedicated, 100% Sharia-compliant supply chain finance and receivables factoring platform designed to streamline cash flows and trade operations.
* **Consultancy & Technology Services:** End-to-end advisory and technical implementation services led by financial technology experts to guide digital transformation.

---

### The Ithmaar Advantage: Why Customers & Investors Choose Us

* **The "No Compromises" Philosophy:** Financial institutions no longer have to choose between strict compliance and modern tech. Ithmaar's modular solutions allow organizations to adopt only the tools they need, minimizing operational disruption while maximizing efficiency.
* **Built for the Modern Financial Landscape:** Our solutions serve Islamic financial institutions, conventional banks operating Islamic windows, microfinance organizations, and enterprise corporate lenders seeking scalable, future-ready infrastructure.
* **Market-Driven Innovation:** With proprietary solutions like Tejara and integrated predictive AI, Ithmaar is positioned at the intersection of high-growth sectors: FinTech, Artificial Intelligence, and ethical/Islamic Finance.

---

### Our People & Culture

At the heart of Ithmaar is a group of seasoned industry veterans who united around a shared vision: revolutionizing how financial institutions scale and serve their communities. 

* **Veteran Expertise:** Our founding team and technical leadership bring decades of cross-disciplinary experience across micro, retail, and corporate Islamic finance, paired with modern enterprise software engineering.
* **A Game-Changing Mindset:** We foster a collaborative, ambitious culture driven by innovation, agility, and integrity. We challenge conventional banking bottlenecks to build robust solutions that redefine the industry standard.
* **Careers at Ithmaar:** For engineers, product visionaries, and financial experts, Ithmaar offers the opportunity to build mission-critical financial infrastructure and shape the global Islamic FinTech revolution. Team members work alongside senior leaders in an impact-driven, high-growth environment.

---

### Connect With Us

Ready to transform your financial operations, explore investment opportunities, or join our growing team?

* **Website:** Visit our platform to request a demo of Ithmaar Finance.
* **Inquiries:** Reach out through our contact portal to discover how Ithmaar can build your future, your way.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>